# Ext_Debt_Data via `lic_dsf.pv`

Goal: recreate **Ext_Debt_Data** by loading new-debt instruments from the template,
aggregating through a portfolio, then folding in existing debt via `ExternalDebtBook`.

| Layer | Class / API | Role | Status |
|-------|-------------|------|--------|
| Instrument | `PresentValueInstrument` | One new loan (`internal` / `external`) | works |
| Load | `load_instruments_from_workbook` | Input 4 terms + disbursements → instruments | works |
| Portfolio | `PVPortfolio` | Owns instruments → new-debt aggregates | works |
| Book | `ExternalDebtBook` | Existing debt + ST/SDR + Ext_Debt headlines | works |

See `docs/04-ext-debt-module-design.qmd`.


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 80)

WORKBOOK

PosixPath('/home/sravan/excel-grapher/lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

## 1. Load every Input 4 / PV_Base instrument from the template

`include_zero_disbursement=True` so the Ext_Debt creditor list is complete (many lines are zero in this template).

```python
instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=True,
)
# each item is a PresentValueInstrument
```

In [2]:
from lic_dsf.pv import load_instruments_from_workbook

instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=True,
)

catalog = pd.DataFrame(
    [
        {
            "name": i.name,
            "grace": i.grace,
            "maturity": i.maturity,
            "interest": i.interest_rate,
            "discount": i.discount_rate,
            "disbursement_sum": sum(i.disbursements),
            "n_years": len(i.disbursements),
        }
        for i in instruments
    ]
)
print(f"{len(instruments)} instruments from {WORKBOOK.name}")
catalog

30 instruments from lic-dsf-template-2025-08-12.xlsx


,name,grace,maturity,interest,discount,disbursement_sum,n_years
0,IMF,5,10,0.0025,0.0500,0.0000,21
1,IDA - regular,6,38,0.0075,0.0500,790.3180,21
2,IDA - 50Y loans,10,50,0.0000,0.0500,300.0000,21
3,IDA - SML,6,12,0.0000,0.0500,10.0000,21
4,IDA NEW 40-year credits,11,40,0.0000,0.0500,10.0000,21
5,IDA NEW Regular,6,31,0.0075,0.0500,100.0000,21
6,IDA NEW Blend (also enter) -->,5,25,0.0324,0.0500,120.0000,21
7,IDA NEW 60-year credits,20,60,0.0000,0.0500,120.0000,21
8,MULTI1,5,30,0.0075,0.0500,0.0000,21
9,MULTI2,5,30,0.0000,0.0500,0.0000,21


## 2. Spot-check one instrument Output (works today once loaded)

Pick a nonzero line (Eurobond in this template) and show the Ext_Debt-facing Output rows: Interest, Amortization, PV, Stock.

In [3]:
by_name = {i.name: i for i in instruments}
sample_name = "Eurobond" if "Eurobond" in by_name else instruments[0].name
sample = by_name[sample_name]

external = sample.external()
ext_debt_rows = external.loc[
    [
        "Interest",
        "Amortization",
        f"PV of debt   {sample.name}",
        "Stock of new forex debt (in USD)",
    ]
]
print(sample_name, "→ Ext_Debt Output metrics")
ext_debt_rows.iloc[:, :10]

Eurobond → Ext_Debt Output metrics


,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
Interest,0.0000,0.0000,0.0000,0.0000,22.5000,45.0000,67.5000,157.5000,187.5000,217.5000
Amortization,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
PV of debt Eurobond,0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3333","2,416.6667","2,750.0000"
Stock of new forex debt (in USD),0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3333","2,416.6667","2,750.0000"


## 3. `PVPortfolio` — own instruments, build new-debt aggregates

```python
portfolio = PVPortfolio(instruments)
portfolio.aggregate_external()
portfolio.interest()
portfolio.amortization()
portfolio.pv()
portfolio.stock()
portfolio.new_debt_service()
```

These are the Ext_Debt **new MLT** panels (Interest / Amortization / PV / Stock totals), before old debt or DSA headlines.

In [4]:
from lic_dsf.pv import PVPortfolio

portfolio = PVPortfolio(tuple(instruments))

totals = portfolio.aggregate_external()
assert isinstance(totals, pd.DataFrame)

display(totals.iloc[:, :10])

interest = portfolio.interest()
amortization = portfolio.amortization()
pv = portfolio.pv()
stock = portfolio.stock()
new_ds = portfolio.new_debt_service()

print("interest shape:", interest.shape)
print("amortization shape:", amortization.shape)
print("pv shape:", pv.shape)
print("stock shape:", stock.shape)
new_ds.iloc[:, :10]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
"New forex borrowing (gross, USD)","1,071.9440",984.3925,"1,210.6787","1,594.3500","1,673.9300","1,787.4800","2,352.8957","1,962.6378","2,241.8368","2,325.6747"
cumulative,"1,071.9440","2,056.3364","3,267.0151","4,861.3651","6,535.2951","8,322.7751","10,675.6709","12,638.3086","14,880.1455","17,205.8202"
Stock of new forex debt (in USD),"1,071.9440","2,056.3364","3,179.5151","4,686.3651","6,245.4861","7,884.9789","10,141.8928","11,965.5436","13,971.4098","15,896.4733"
PV of debt,899.1668,"1,620.9792","2,360.3225","3,495.0002","4,822.3646","6,319.3737","8,559.8699","10,313.0252","12,260.7004","14,139.9341"
Total debt service (in USD),0.0000,37.5339,150.0686,171.3496,248.3110,342.8919,363.9449,550.2004,735.8611,999.0116
Interest,0.0000,37.5339,62.5686,83.8496,133.5020,194.9047,267.9632,411.2133,499.8905,598.4003
Amortization,0.0000,0.0000,87.5000,87.5000,114.8090,147.9873,95.9818,138.9870,235.9706,400.6112


interest shape: (30, 61)
amortization shape: (30, 61)
pv shape: (30, 61)
stock shape: (30, 61)


,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
Interest,0.0000,37.5339,62.5686,83.8496,133.5020,194.9047,267.9632,411.2133,499.8905,598.4003
Amortization,0.0000,0.0000,87.5000,87.5000,114.8090,147.9873,95.9818,138.9870,235.9706,400.6112
Total new debt service,0.0000,37.5339,150.0686,171.3496,248.3110,342.8919,363.9449,550.2004,735.8611,999.0116


## 4. `ExternalDebtBook` — Ext_Debt_Data headlines

Wires `PVPortfolio` new-MLT aggregates to **existing** MLT NPV (Excel: "old"),
arrears, ST, SDR, and headline totals (`total_pv_of_debt`, public debt service,
PPG check). Fold LC-NR into the portfolio for full new-MLT parity with Ext R279.


In [5]:
from lic_dsf.pv import (
    ExternalDebtBook,
    load_external_debt_inputs,
    load_lc_nr_instruments_from_workbook,
)

lc_nr = load_lc_nr_instruments_from_workbook(WORKBOOK)
book_portfolio = PVPortfolio(tuple(instruments) + tuple(lc_nr))
book = ExternalDebtBook(
    portfolio=book_portfolio,
    inputs=load_external_debt_inputs(WORKBOOK),
)
book.summary().iloc[:, :10]


,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032
PV of existing MLT debt,"18,211.9607","16,244.3877","14,408.7000","13,067.7952","11,765.2646","10,295.2596","9,329.7091","7,020.3270","6,008.2588","5,081.8812"
PV of existing arrears,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
PV of new MLT debt,0.0000,"4,061.9963","6,833.1360","8,725.8791","10,910.2984","13,040.5630","15,219.9064","17,833.6104","18,818.5881","20,653.5132"
Total ST external debt,150.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
of which: locally-issued ST,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
PV of net use of SDRs,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Total PV of debt,"18,361.9607","20,406.3840","21,241.8360","21,793.6743","22,675.5630","23,335.8227","24,549.6154","24,853.9374","24,826.8469","25,735.3944"
Nominal value of new MLT,0.0000,"4,234.7734","7,268.4932","9,545.0717","12,101.6633","14,463.6845","16,785.5115","19,415.6333","20,471.1065","22,364.2226"
Nominal PPG debt check,0.0000,-0.0000,0.0000,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.0000,0.0000
Grant element of new disbursements (%),0.0000,3.9147,7.3828,10.0706,9.4688,7.0995,5.0441,2.1485,3.6520,3.3246


## 5. Locally-issued debt (Input 5)

`load_external_debt_inputs` converts Input 5 LC stock / principal / interest / ST
to USD with Macro FX(eop)/FX(pa). Those series feed public debt service and
total ST on `ExternalDebtBook` (Excel R56 / R58 / R386 / R394–396).

In [6]:
local = pd.DataFrame(
    {
        "stock": book.inputs.locally_issued_debt_stock,
        "principal": book.inputs.locally_issued_principal,
        "interest": book.inputs.locally_issued_interest,
        "st": book.inputs.locally_issued_st,
    }
)
display(local.loc[2024:2030])
book.total_public_debt_service().iloc[:, :8]

,stock,principal,interest,st
2024,"1,700.5981",807.7790,559.1002,0.0000
2025,871.6805,722.1637,330.5335,0.0000
2026,545.3133,279.8592,177.3420,0.0000
2027,414.3340,107.9101,125.8848,0.0000
2028,57.8176,347.3621,96.5851,0.0000
2029,55.6599,0.0000,10.7805,0.0000
2030,53.5827,0.0000,10.3782,0.0000


,2023,2024,2025,2026,2027,2028,2029,2030
Total public debt service,"2,482.8964","3,212.2153","3,388.9257","3,320.9447","3,749.8586","4,545.0486","4,163.3776","5,462.8687"
of which: principal,"1,674.4184","2,195.6931","1,920.2255","1,701.5632","2,065.8817","2,905.4316","2,581.3700","3,108.9647"
of which: interest,808.4780,"1,016.5222","1,468.7002","1,619.3815","1,683.9769","1,639.6170","1,582.0076","2,353.9040"


## 6. Grant element of new disbursements

Ext `R408` is the disbursement-weighted average of each instrument's unit-loan
grant element % (from `PresentValueInstrument.internal()`, same value Input 4
column I links to). `R407` / `R409` follow from that average and total new
external MLT disbursements.

In [7]:
ge = pd.DataFrame(
    {
        "GE % (R408)": book.grant_element_percent(),
        "disb net of GE (R407)": book.new_disbursements_net_of_ge(),
        "GE value (R409)": book.grant_element_value(),
    }
)
ge.loc[2024:2030]

,GE % (R408),disb net of GE (R407),GE value (R409)
2024,3.9147,"4,240.7757",172.7771
2025,7.3828,"3,131.6608",249.6351
2026,10.0706,"2,727.0722",305.3878
2027,9.4688,"3,300.7632",345.2308
2028,7.0995,"3,703.2339",283.0045
2029,5.0441,"3,928.7524",208.6972
2030,2.1485,"4,300.3691",94.4232


## 7. New debt by creditor group

Ext rolls new disbursements / interest / amort / PV / stock into six creditor
groups (Multilaterals → FX local residents). `PresentValueInstrument` is
unchanged — the book only `groupby`s portfolio Output rows.

In [8]:
book.new_interest_by_creditor().loc[:, 2024:2030]

,2024,2025,2026,2027,2028,2029,2030
Multilaterals,0.0000,1.1250,1.9230,4.8292,7.5654,9.0654,10.5654
Other multilaterals,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Official bilaterals,0.0000,6.6199,15.9263,25.4061,37.4044,47.2975,53.0054
Commercial,0.0000,29.7890,44.7193,53.6142,88.5322,138.5418,204.3924
Locally issued (NR),0.0000,454.7350,733.9042,876.4172,965.4547,992.3069,994.4220
FX local (residents),0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Total,0.0000,492.2689,796.4728,960.2668,"1,098.9568","1,187.2116","1,262.3852"


## 8. Existing service, evolution, memorandum

Reshape of data already on the book: per-creditor existing service, Ext R42–R63
service headlines, stock evolution (R45 / R58 / R67), and memorandum outstanding
+ FX (R398 / R402 / R403).

In [9]:
display(book.existing_service_totals().loc[:, 2024:2030])
display(book.debt_evolution().loc[:, 2024:2030])
book.memorandum().loc[:, 2024:2030]

,2024,2025,2026,2027,2028,2029,2030
Existing external debt service,"1,680.3361","1,733.9595","1,691.3886","1,797.6754","1,681.0351","1,475.2650","2,771.0073"
Existing principal,"1,237.9141","1,098.0617","1,045.8220","1,199.8500","1,236.9600","1,091.2495","1,689.8667"
Existing interest,442.4220,635.8977,645.5667,597.8253,444.0752,384.0155,"1,081.1406"
Locally-issued debt service,"1,366.8792","1,052.6972",457.2012,233.7950,443.9472,10.7805,10.3782
Locally-issued principal,807.7790,722.1637,279.8592,107.9101,347.3621,0.0000,0.0000
Locally-issued interest,559.1002,330.5335,177.3420,125.8848,96.5851,10.7805,10.3782
Total existing + local service,"3,047.2153","2,786.6567","2,148.5898","2,031.4703","2,124.9823","1,486.0455","2,781.3855"
Total principal,"2,045.6931","1,820.2255","1,325.6811","1,307.7602","1,584.3221","1,091.2495","1,689.8667"
Total interest,"1,001.5222",966.4313,822.9087,723.7102,540.6602,394.7960,"1,091.5188"


,2024,2025,2026,2027,2028,2029,2030
Existing external (excl. local),"15,483.4874","14,385.4256","13,339.6037","12,139.7536","10,902.7937","9,811.5442","8,121.6775"
Locally-issued,"1,700.5981",871.6805,545.3133,414.3340,57.8176,55.6599,53.5827
Existing MLT (incl. local adj.),"17,184.0855","15,257.1061","13,884.9169","12,554.0876","10,960.6112","9,867.2041","8,175.2602"


,2024,2025,2026,2027,2028,2029,2030
External debt outstanding,"21,418.8589","22,525.5993","23,429.9887","24,655.7509","25,424.2958","26,652.7156","27,590.8934"
Exchange rate (eop),4.7033,5.1105,5.4574,5.7263,5.9527,6.1834,6.4232
Exchange rate (pa),4.4516,4.9069,5.2840,5.5918,5.8395,6.0681,6.3033


## 9. Residual financing params (Input 7)

Ext_Debt computes decade averages (`AVERAGE(F:P)` → `C126–C128`, `C131–C133`) that Input 7 uses as **Default** shares/terms. Override any field; unresolved fields keep the default. Partial share overrides renormalize domestic ST to `1 - external - domestic_mlt` (Input 7 public-DSA `I11`).

```python
defaults = book.residual_defaults()
resolved = book.residual_params(
    ResidualFinancingOverrides(avg_interest_rate=6.0),
)
```

In [10]:
from dataclasses import asdict

from lic_dsf.pv import ResidualFinancingOverrides

defaults = book.residual_defaults()
resolved = book.residual_params(
    ResidualFinancingOverrides(
        external_mlt_share=0.5,
        domestic_mlt_share=0.2,
        avg_interest_rate=6.0,
    )
)

pd.DataFrame(
    {
        "default": asdict(defaults),
        "resolved": asdict(resolved),
    }
)

,default,resolved
external_mlt_share,0.4344,0.5000
domestic_mlt_share,0.2276,0.2000
domestic_st_share,0.3371,0.3000
avg_interest_rate,7.9824,6.0000
avg_grace,4.1378,4.1378
avg_maturity,9.5804,9.5804
avg_grace_rounded,4.0000,4.0000
avg_maturity_rounded,9.0000,9.0000


## Build order

1. `load_instruments_from_workbook` — Input 4 / PV_Base as `PresentValueInstrument`
2. `PVPortfolio` — new-debt aggregates
3. `ExternalDebtBook` — existing MLT NPV + ST/SDR + Ext headlines
4. Locally-issued Input 5 series on the book (USD)
5. Grant element of new disbursements (`grant_element_percent`)
6. Creditor-group panels (`new_*_by_creditor`)
7. Existing service / evolution / memorandum panels
8. Residual financing params (`residual_defaults` / `residual_params`)
9. Later: stress DSA, Ext R399 FX-denominated path, Dom_Debt_Data
